# [LAB] RAG 실습

보험 약관 PDF 한 권으로 RAG를 처음부터 끝까지 만들어 봅니다. 장마다 RAG 핵심 모듈을 하나씩 더합니다.

- **2장 PDF 불러오기와 청크 나누기** (과제): Loading과 Splitting입니다. PDF를 읽고 청크로 나눕니다.
- **3장 임베딩과 유사도** (예제): Embedding입니다. 문장을 벡터로 바꾸고 뜻이 가까운 정도를 숫자로 비교합니다.
- **4장 벡터 저장소와 검색** (과제): Storage와 Retrieval입니다. 청크를 Chroma에 넣고 질문과 가까운 청크를 찾습니다.
- **5장 RAG 체인** (과제): Generation입니다. 검색 결과를 근거로 답변하는 체인을 완성하고 스트리밍으로 받습니다.
- **6장 RAGaaS** (예제): 청킹·임베딩·저장을 API가 대신 하는 방식으로 같은 일을 합니다.

셀은 위에서부터 차례로 실행합니다. 각 장 첫머리의 접힌 준비 코드도 실행해야 그 아래 셀이 동작합니다.

In [ ]:
%cd ../../../course/Chapter01/notebooks

**목차**

- 1. 실습 방법
- 2. PDF 불러오기와 청크 나누기 (예제) (과제)
  - 2.1 PDF를 Document로 불러오기 (과제)
  - 2.2 Document를 청크로 나누기 (과제)
  - 2.3 청크 확인하기 (예제)
- 3. 임베딩과 유사도 (예제)
  - 3.1 사내 임베딩을 LangChain에 맞추기 (예제)
  - 3.2 문장을 벡터로 바꾸기 (예제)
  - 3.3 유사도 비교와 질문 검색 (예제)
  - 3.4 AI에게 물어보기
- 4. 벡터 저장소와 검색 (예제) (과제)
  - 4.1 벡터 저장소 만들기 (과제)
  - 4.2 retriever로 바꾸기 (과제)
  - 4.3 질문으로 청크 찾기 (예제)
- 5. RAG 체인 (예제) (과제)
  - 5.1 벡터 저장소 만들기 (과제)
  - 5.2 retriever로 바꾸기 (과제)
  - 5.3 모델과 프롬프트 준비 (예제)
  - 5.4 검색 결과를 context로 합치기 (과제)
  - 5.5 RAG 체인 연결하기 (과제)
  - 5.6 질문하고 스트리밍으로 답변 받기 (예제)
- 6. RAGaaS (예제)
  - 6.1 1부: 업로드할 문서 준비 (예제)
  - 6.2 2부: 문서 검색 (예제)
  - 6.3 3부: 검색 결과로 답변 만들기 (예제)
  - 6.4 4부: 문서 삭제(실행하지 않음) (예제)
  - 6.5 AI에게 물어보기
- 7. 정리

In [ ]:
from config.check import check_env

# 값은 출력하지 않고, 필요한 설정 이름이 있는지만 봅니다
check_env(["EMBED_PROVIDER", "RAG_PROVIDER"])

**볼 지점**
- "준비 완료"가 나오면 다음으로 넘어갑니다.
- "준비가 덜 되었습니다"가 나오면 ⛔ 줄의 → 안내대로 `.env`를 고친 뒤 이 셀을 다시 실행합니다. 무엇을 고칠지 모르겠으면 이 셀의 출력을 AI에게 붙여 넣고 물어봅니다.
- 키 값은 화면에 나오지 않습니다. 혹시 나오면 셀 출력을 지우고, 그 내용을 다른 곳에 붙여 넣지 않습니다.

<details><summary>강사용 메모</summary>

- 원본: `src/raw/Chapter01/03-rag/` 다섯 파일. 빌드할 때 셀로 옮겨 넣으므로 수강생은 .py 파일을 보지 않습니다. 약관 PDF는 `lessons/Chapter01/data/meritz_pet_insurance.pdf`를 씁니다(노트북 기준 `data/…`).
- 맨 위 환경 확인 셀은 공용 셀이라 LLM 설정 이름만 봅니다. 임베딩(`EMBED_PROVIDER`, 사외 `OPENAI_EMBEDDING_MODEL`, 사내 `embedding_api_url`·`embedding_x_dep_ticket`)과 RAGaaS(`RAG_PROVIDER`, `rag_api_key`, `rag_credential_key`, `search_url`) 설정이 빠지면 3장 이후 준비 코드에서 `RuntimeError`로 멈춥니다.
- 네 장(2·4·5·6)이 약관 PDF를 다시 읽고, 4·5장은 청크 50개를 다시 임베딩합니다(사내 임베딩은 10개씩 5회 요청).
- CodeMate가 노트북 셀을 읽고 고칠 수 있는지는 강의장에서 미확인입니다. 못 읽으면 셀 코드를 지시문 아래에 붙여 넣게 합니다.

</details>

## 1. 실습 방법

실습은 두 종류입니다. 제목 옆 표시와 코드 주석으로 구분합니다.

- **예제** `# ✍️ CHANGE_HERE:` 바로 아래 줄의 값을 바꿔 다시 실행해 봅니다.
- **과제** `# ✍️ WRITE_HERE:` 아래 빈칸(`...`)을 AI와 함께 채웁니다. **프롬프트 힌트**를 보고 지시를 직접 씁니다.

실행한 뒤에는 **성공 기준**과 비교해 스스로 판단합니다. 🟢는 되어야 하는 것, ⛔는 다시 해야 하는 신호입니다. 과제는 그다음 접힌 "정답 보기"와 비교합니다.

**막혔을 때: 스스로 먼저 확인합니다**

1. 오류가 난 셀과 오류의 마지막 줄을 AI에게 붙여 넣고, 원인과 고칠 곳을 물어봅니다. 처음 한 번은 "저장소 맨 위 AGENTS.md를 읽고 도와줘"라고 먼저 말합니다. AI가 설치·경로·연결 문제의 해결 순서를 알게 됩니다.
2. 설정 문제로 보이면 맨 위 환경 점검 셀을 다시 실행하고, ⛔ 줄의 → 안내대로 고칩니다.
3. 커널을 다시 시작하고 맨 위 셀부터 다시 실행합니다.
4. 옆 사람 화면과 비교해 봅니다.
5. 그래도 안 되면 오류 화면을 그대로 두고 강사에게 보여 줍니다. 그동안 그 셀은 건너뛰고 다음 절을 진행해도 됩니다.

## 2. PDF 불러오기와 청크 나누기 <small>(예제) (과제)</small>

약관 PDF를 읽어 `Document`로 만들고, 검색하기 좋은 크기의 청크로 나눕니다. RAG 핵심 모듈 중 Loading과 Splitting입니다. 이 장은 API를 호출하지 않습니다.

빈칸 두 곳을 AI와 함께 채우고 차례로 실행합니다. 그다음 `# ✍️ CHANGE_HERE:` 줄의 청크 크기를 바꿔 결과가 어떻게 달라지는지 봅니다.

**왜 배우나:** 청크 크기는 검색 정확도와 문맥 보존 사이의 균형을 정합니다. 뒤 장의 검색 결과는 모두 여기서 나눈 청크에서 나옵니다.

준비 코드입니다. PDF를 읽는 도구와 나누는 도구를 불러오고, 약관 PDF 파일 경로(`PDF_PATH`)를 정합니다. 펼쳐 보지 않고 실행만 해도 됩니다.

In [ ]:
"""
데이터 로딩과 청킹: PDF를 Document와 Chunk로 변환하기

핵심 개념:
  - Document = LangChain이 문서 본문과 메타데이터를 다루는 기본 단위
  - Chunking = 긴 문서를 검색 가능한 작은 조각으로 나누는 과정
  - chunk_size와 chunk_overlap은 검색 정확도와 문맥 보존 사이의 균형을 조정

"""

from pathlib import Path

import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


PDF_PATH = Path("data/meritz_pet_insurance.pdf")

<details><summary>강사용 메모</summary>

- 원본 `01_data_loading.py`(raw)를 쓰고, 노트북에서만 세 곳을 손봤습니다. ① 독스트링의 `실행: python 03-rag/…` 줄 삭제 ② `PDF_PATH`를 `Path(__file__)` 대신 `data/…` 상대 경로로 ③ `langchain-community` 종료 예고 경고(DeprecationWarning, 외부 주소 포함)를 준비 코드에서 숨김.
- 결과 출력은 솔루션의 `text[:120]`을 씁니다. raw는 전문을 출력하면서 "`[:120]` 지우기" 주석이 달려 있어 코드와 주석이 어긋납니다.
- `PDFMinerLoader` 기본값은 28쪽 약관을 `Document` 1개로 읽습니다. `metadata`에 쪽 번호가 없습니다(`source`, `total_pages` 등만 있음). 사외 실행 결과 청크는 50개였습니다.

</details>

### 2.1 PDF를 Document로 불러오기 <small>(과제)</small>

**✍️ WRITE_HERE: PDFMinerLoader로 PDF를 읽고 Document 목록을 생성해 주세요.**

PDF 파일을 읽어 `Document` 목록을 만듭니다. `Document` 하나에는 본문(`page_content`)과 메타데이터(`metadata`)가 들어 있습니다. 파일 경로(`PDF_PATH`)와 읽는 도구(`PDFMinerLoader`)는 준비 코드에 있습니다.

**프롬프트 힌트** `PDF_PATH`의 파일을 `PDFMinerLoader`로 읽어 `documents`라는 이름의 `Document` 목록으로 만들어 달라고 합니다. 이 셀의 빈칸 한 곳만 채우고 다른 줄은 고치지 말라고 덧붙입니다.

**성공 기준**

- 🟢 셀이 오류 없이 끝난다(이 셀은 아무것도 출력하지 않음)
- ⛔ `NameError`가 난다 → 준비 코드를 먼저 실행했는지 봅니다
- ⛔ `FileNotFoundError`가 난다 → 파일 경로를 `PDF_PATH`가 아닌 값으로 바꾸지 않았는지 봅니다

In [ ]:
# ── PDF 로딩 ─────────────────────────────────────────────────────────────────
# PDFMinerLoader: PDF 파일을 LangChain Document 목록으로 변환
# 각 Document는 page_content와 metadata를 가진다.
# ✍️ WRITE_HERE: PDFMinerLoader로 PDF를 읽고 Document 목록을 생성해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()  # 결과는 list[Document]
```

</details>

### 2.2 Document를 청크로 나누기 <small>(과제)</small>

**✍️ CHANGE_HERE: 청크 크기를 바꿔 봅니다**

먼저 아래 빈칸을 채워 이 셀과 청크 확인 셀까지 실행합니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 `500`을 `200`처럼 작은 값으로 바꾸고, 이 셀과 청크 확인 셀을 다시 실행합니다. `len=` 값이 어떻게 바뀌는지 봅니다.

**✍️ WRITE_HERE: Text Splitter를 사용해 Document 목록을 Chunk로 나누어 주세요.**

바로 위에서 만든 `text_splitter`로 `documents`를 청크로 나눕니다. `chunk_size`는 청크 하나의 최대 길이, `chunk_overlap`은 이웃한 청크끼리 겹쳐 두는 길이입니다.

**프롬프트 힌트** 위에서 만든 `text_splitter`로 `documents`를 나눠 `chunks`에 담아 달라고 합니다. 설정값(`chunk_size`, `chunk_overlap`)은 바꾸지 말라고 적습니다.

**성공 기준**

- 🟢 오류 메시지 없이 실행이 끝난다(출력 없음)
- ⛔ `NameError: name 'documents'`가 난다 → 앞 셀의 빈칸을 채워 먼저 실행합니다

In [ ]:
# ── 문서 청킹 ─────────────────────────────────────────────────────────────────
# 전체 문서는 한 번에 LLM에 넣기 어렵기 때문에 검색 가능한 Chunk로 나눈다.
# RecursiveCharacterTextSplitter: 문단 → 줄 → 단어 → 문자 순서로 분할 지점을 찾는다.
# chunk_size: 각 Chunk의 최대 길이
# chunk_overlap: 인접 Chunk 사이에 겹쳐 둘 길이
text_splitter = RecursiveCharacterTextSplitter(
    # 작을수록 검색 단위가 정밀하지만 전체 문맥이 줄어든다.
    # ✍️ CHANGE_HERE: 청크 크기를 바꿔 봅니다
    chunk_size=500,
    # 클수록 문맥 연결성이 좋아지지만 중복과 비용이 늘어난다.
    chunk_overlap=75,
)
# ✍️ WRITE_HERE: Text Splitter를 사용해 Document 목록을 Chunk로 나누어 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
chunks = text_splitter.split_documents(documents)
```

</details>

### 2.3 청크 확인하기 <small>(예제)</small>

In [ ]:
# ── 결과 확인 ─────────────────────────────────────────────────────────────────
for index, chunk in enumerate(chunks[:3]):
    text = " ".join(chunk.page_content.split())
    print(f"{index}. len={len(chunk.page_content)} \n- {text[:120]}...")
    print()

**성공 기준**

- 🟢 청크 3개가 `0. len=…`, `1. len=…`, `2. len=…` 모양으로 출력되고, 줄마다 아래에 약관 문장이 보인다
- 🟢 `chunk_size`를 작게 바꿔 다시 실행하면 `len=` 값도 그 값 이하로 작아진다

**볼 지점**
- `len=` 뒤의 숫자는 청크의 글자 수입니다. `chunk_size`보다 크지 않습니다.
- 청크 내용은 앞 120자만 보여 줍니다.
- 원래 값(`500`)과 바꾼 값의 결과를 비교해, 청크가 작아지면 한 청크에 담긴 내용이 얼마나 줄어드는지 봅니다.

## 3. 임베딩과 유사도 <small>(예제)</small>

문장을 임베딩(의미를 숫자 목록, 곧 벡터로 바꾸는 일)하고, 벡터끼리 얼마나 가까운지 코사인 유사도로 비교합니다. RAG에서 "관련 문서 검색"은 이 벡터 거리를 계산하는 일입니다.

코드는 완성되어 있습니다. 차례로 실행하고, 마지막 셀의 `# ✍️ CHANGE_HERE:` 줄에서 질문을 바꿔 봅니다.

첫 셀의 `SamsungEmbeddings`는 사내 임베딩 API를 LangChain에서 쓸 수 있게 맞춘 클래스입니다. LangChain의 `Embeddings`를 물려받아 `embed_documents`(여러 문장)와 `embed_query`(질문 하나) 두 메서드만 만들면, 벡터 저장소나 검색기 같은 LangChain 도구에서 그대로 쓸 수 있습니다. 셀 아래쪽의 `if`는 연결 환경에 맞는 임베딩을 고르는 코드라 건너뛰어도 됩니다.

준비 코드입니다. 필요한 도구를 불러오고 설정 파일을 읽습니다. 내용은 몰라도 되니 실행만 하고 넘어갑니다.

In [ ]:
"""
임베딩(Embedding): 텍스트를 숫자 벡터로 변환하기

핵심 개념:
  - 임베딩 = 텍스트의 '의미'를 숫자 벡터로 표현
  - 의미가 비슷한 텍스트는 벡터 공간에서 가까이 위치
  - RAG에서 '관련 문서 검색'은 이 벡터 거리를 계산하는 것

"""

import os

import numpy as np
from dotenv import load_dotenv
import config  # noqa: F401  .env의 COURSE_ENV(사내/사외)에 맞춰 연결 방식을 채운다
from config.embeddings import get_embeddings
from config.samsung import embedding_config

load_dotenv()

import requests
from langchain_core.embeddings import Embeddings

<details><summary>강사용 메모</summary>

- 원본 `02_embeddings.py`(raw)를 씁니다. 벡터 출력은 raw의 `v[:5]`(앞 5개 값)입니다. 솔루션은 벡터 전체(수천 개 숫자)를 출력합니다.
- 노트북에서만 손본 곳: ① 독스트링 `실행:` 줄 삭제 ② 클래스 안의 `config/samsung.py` 경로 주석 삭제 ③ 질문을 `query` 변수로 뺌. 원본은 `embed_query(…)`와 `print(…)`에 같은 질문을 두 번 적어서, 한쪽만 바꾸면 출력이 조용히 틀립니다 ④ 원본 끝의 "실습: SamsungEmbeddings 구현" 구획은 주석만 있고 코드가 없어 뺐습니다. 내용: 사내 임베딩 API는 LangChain과 직접 호환되지 않으니 `Embeddings`를 상속해 두 메서드만 구현하면 vectorstore·retriever에서 그대로 쓸 수 있다.
- 갈림길: `EMBED_PROVIDER=samsung`(기본)이면 `SamsungEmbeddings`(사내 `embedding_config()` 필요), 그 밖이면 `get_embeddings()`(사외 OpenAI 임베딩).
- 세 번째 문장을 바꿔 보는 CHANGE_HERE는 넣지 않았습니다. 출력 이름표 `[주식]`이 코드에 고정돼 있어, 문장만 바꾸면 이름표와 내용이 어긋납니다.
- 유사도 숫자는 모델마다 달라 정답으로 고정하지 않습니다. 사내 임베딩의 벡터 차원·유사도 값은 미확인입니다.

</details>

### 3.1 사내 임베딩을 LangChain에 맞추기 <small>(예제)</small>

In [ ]:
BATCH = 10  # API 한 번에 처리 가능한 최대 텍스트 수


class SamsungEmbeddings(Embeddings):
    def embed_documents(self, texts):
        # BATCH 개씩 나눠서 API 호출, 결과를 하나로 합침
        result = []
        for i in range(0, len(texts), BATCH):
            config = embedding_config()
            res = requests.post(
                config["url"],
                json={"input": texts[i : i + BATCH]},
                headers=config["headers"],
                timeout=30,
            )
            res.raise_for_status()
            result.extend(item["embedding"] for item in res.json()["data"])
        return result

    def embed_query(self, text):
        # 질문 하나를 임베딩할 때는 embed_documents 재활용
        return self.embed_documents([text])[0]


provider = os.getenv("EMBED_PROVIDER", "samsung").strip().lower()
if provider == "samsung":
    embedding_config()  # 설정이 비어 있으면 여기서 멈춘다.
    embeddings = SamsungEmbeddings()
else:
    # 사외에서는 같은 유사도 실습을 OpenAI 임베딩으로 실행한다.
    embeddings = get_embeddings()

### 3.2 문장을 벡터로 바꾸기 <small>(예제)</small>

In [ ]:
sentences = [
    "강아지가 공원에서 뛰어놀고 있다.",
    "개가 잔디밭에서 달리고 있다.",
    "오늘 주식시장이 크게 하락했다.",
]

# embed_documents(): 텍스트 리스트 → 벡터 리스트
vectors = embeddings.embed_documents(sentences)

print("=== 벡터 기본 정보 ===")
print(f"문장 수    : {len(vectors)}")
print(f"벡터 차원  : {len(vectors[0])}")
print(f"앞 5개 값  : {[v[:5] for v in vectors]}")
print()

**성공 기준**

- 🟢 `문장 수`가 3이고, `벡터 차원`에 0보다 큰 수가 출력된다
- 🟢 `앞 5개 값`에 문장마다 숫자 5개씩 묶음이 보인다

### 3.3 유사도 비교와 질문 검색 <small>(예제)</small>

**✍️ CHANGE_HERE: 질문을 바꿔 봅니다**

먼저 그대로 실행해 가장 유사한 문장을 봅니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 따옴표 안 질문을 바꿔 다시 실행합니다(예: "주가가 떨어졌다"). 따옴표는 지우지 않습니다.

In [ ]:
# ── 코사인 유사도 ─────────────────────────────────────────────────────────────
# 코사인 유사도: 1에 가까울수록 의미가 유사, 0에 가까울수록 관련 없음
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))


v = [np.array(vec) for vec in vectors]
print("=== 문장 간 유사도 ===")
print(f"[강아지] vs [개]   : {cosine_similarity(v[0], v[1]):.4f}")  # 높아야 함
print(f"[강아지] vs [주식] : {cosine_similarity(v[0], v[2]):.4f}")  # 낮아야 함
print()

# embed_query(): 질문 하나를 벡터로 변환 (검색 시 사용)
# ✍️ CHANGE_HERE: 질문을 바꿔 봅니다
query = "애완동물이 뛰노는 모습"
query_vec = np.array(embeddings.embed_query(query))
scores = [cosine_similarity(query_vec, v_i) for v_i in v]
best = scores.index(max(scores))
print(f'질문: "{query}"')
print(f'가장 유사한 문장: "{sentences[best]}" (유사도: {scores[best]:.4f})')
print()

**성공 기준**

- 🟢 `[강아지] vs [개]`의 값이 `[강아지] vs [주식]`의 값보다 크다
- 🟢 질문을 바꿔 다시 실행하면 `질문:` 줄에 바꾼 질문이 보이고, `가장 유사한 문장`이 그 질문과 뜻이 가까운 문장으로 바뀐다

**볼 지점**
- `벡터 차원`은 문장 하나를 바꾼 숫자 목록의 길이입니다.
- 유사도는 1에 가까울수록 뜻이 비슷하고, 0에 가까울수록 관련이 없습니다.
- 유사도 숫자는 임베딩 모델마다 다릅니다. 숫자 자체보다 어느 쪽이 더 큰지를 봅니다.
- 질문 "애완동물이 뛰노는 모습"에는 "강아지"나 "개"라는 낱말이 없습니다. 그래도 뜻이 가까운 문장을 찾는지 봅니다.

### 3.4 AI에게 물어보기

**프롬프트 힌트** `SamsungEmbeddings` 클래스가 든 셀을 붙여 넣고, 두 메서드(`embed_documents`, `embed_query`)가 각각 무엇을 받아 무엇을 돌려주는지 설명해 달라고 합니다. 이 두 메서드만으로 LangChain 도구에서 쓸 수 있는 이유도 묻고, 코드는 고치지 말라고 적습니다.

**성공 기준**

- 🟢 `embed_documents`는 문장 목록을 받아 벡터 목록을, `embed_query`는 문장 하나를 받아 벡터 하나를 돌려준다고 짚었다
- 🟢 LangChain의 `Embeddings`를 물려받아 메서드 이름을 맞췄기 때문이라고 설명했다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다
- ⛔ `BATCH`로 10개씩 나눠 요청하는 반복문을 설명에서 빠뜨렸다 → 반복문이 왜 있는지 다시 묻습니다

## 4. 벡터 저장소와 검색 <small>(예제) (과제)</small>

2장에서 나눈 청크를 임베딩해 Chroma 벡터 저장소에 넣고, 질문과 가까운 청크 3개를 찾아봅니다. RAG 핵심 모듈 중 Embedding, Storage, Retrieval입니다. PDF를 읽고 나누는 코드는 준비 코드에 들어 있습니다.

빈칸 두 곳을 AI와 함께 채우고, 마지막 셀에서 `# ✍️ CHANGE_HERE:` 줄의 질문을 바꿔 검색해 봅니다.

**왜 배우나:** 검색 결과가 틀리면 생성 단계가 좋아도 답변이 흔들립니다. 답변을 만들기 전에 검색 결과만 따로 확인해 봅니다.

준비 코드입니다. 필요한 도구를 불러오고, 2장과 같은 방법으로 약관 PDF를 읽어 청크(`chunks`)로 나눕니다. 접혀 있으니 펼치지 않아도 됩니다. 실행만 합니다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings

load_dotenv()


PDF_PATH = Path("data/meritz_pet_insurance.pdf")
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

<details><summary>강사용 메모</summary>

- 원본 `03_vector_store.py`(raw)를 쓰고, 노트북에서만 손봤습니다. ① `PDF_PATH` 상대 경로, 경고 숨김 ② `src/config/embeddings.py`·`02_embeddings.py`를 가리키는 주석, Chroma·retriever 인자 사이의 메모 주석, `persist_directory` 주석 코드 삭제(정답은 솔루션 모양) ③ `while True` + `input()` 반복을 질문 변수 하나로 한 번만 실행(사용자 결정). 질문은 원본 05의 질문을 씁니다.
- `Chroma().delete_collection()` 한 줄을 더했습니다. 한 커널에서 `Chroma.from_documents`를 다시 호출하면 같은 메모리 컬렉션(`langchain`)에 청크가 쌓여(50 → 100) 검색 결과에 같은 청크가 두 번 나옵니다(사외 실행으로 확인). 원본은 파일마다 새 프로세스라 이 문제가 없습니다.
- 저장소는 메모리에만 있어 파일이 생기지 않습니다.

</details>

### 4.1 벡터 저장소 만들기 <small>(과제)</small>

**✍️ WRITE_HERE: 임베딩 모델을 가져오고 Chroma 벡터스토어를 생성해 주세요.**

청크를 벡터로 바꿔 Chroma 벡터 저장소에 넣습니다. 임베딩 모델은 준비 코드에서 불러온 `get_embeddings()`로 가져옵니다. 셀 첫 줄의 `Chroma().delete_collection()`은 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게 저장소를 비우는 줄이라 지우지 않습니다.

**프롬프트 힌트** `get_embeddings()`로 임베딩 모델을 `embeddings`에 받고, `chunks`로 Chroma 벡터 저장소를 만들어 `vectorstore`에 담아 달라고 합니다. 저장소는 파일로 남기지 말고 메모리에만 만들라고 적습니다.

**성공 기준**

- 🟢 오류 없이 끝난다(청크를 모두 임베딩하느라 몇 초에서 수십 초 걸릴 수 있음)
- ⛔ `RuntimeError`에 `환경변수가 필요합니다`가 보인다 → 맨 위 환경 점검 셀을 다시 실행해 빠진 이름을 확인하고 `.env`에 채운 뒤, 커널을 다시 시작합니다
- ⛔ `NameError: name 'chunks'`가 난다 → 이 장의 준비 코드를 먼저 실행합니다

In [ ]:
# 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게, 앞서 만든 벡터 저장소를 비웁니다.
Chroma().delete_collection()

# ✍️ WRITE_HERE: 임베딩 모델을 가져오고 Chroma 벡터스토어를 생성해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
embeddings = get_embeddings()
vectorstore = Chroma.from_documents(
    chunks,
    embedding=embeddings,
)
```

</details>

### 4.2 retriever로 바꾸기 <small>(과제)</small>

**✍️ WRITE_HERE: 벡터스토어를 retriever로 변환해 주세요.**

벡터 저장소를 검색기(retriever)로 바꿉니다. retriever는 질문을 받으면 가까운 청크를 찾아 돌려주는 객체입니다. 한 번에 가져올 청크 수는 3개입니다.

**프롬프트 힌트** `vectorstore`를 질문과 가까운 청크 3개를 돌려주는 retriever로 바꿔 `retriever`에 담아 달라고 합니다. 검색을 실행하거나 출력하는 코드는 넣지 말라고 적습니다.

**성공 기준**

- 🟢 빨간 오류 메시지 없이 셀 실행이 끝난다(출력 없음)
- ⛔ `NameError: name 'vectorstore'`가 난다 → 앞 셀의 빈칸을 채워 먼저 실행합니다

In [ ]:
# ✍️ WRITE_HERE: 벡터스토어를 retriever로 변환해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

</details>

### 4.3 질문으로 청크 찾기 <small>(예제)</small>

**✍️ CHANGE_HERE: 질문을 바꿔 봅니다**

빈칸 두 곳을 채워 실행한 뒤 이 셀을 실행합니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 약관에 있을 법한 다른 질문(예: "보험금을 청구하려면 무엇이 필요해?")으로 바꿔 다시 실행합니다.

In [ ]:
print("retriever ready")
# ✍️ CHANGE_HERE: 질문을 바꿔 봅니다
question = "펫보험 보장 내용이 뭐야?"
results = retriever.invoke(question)
for index, doc in enumerate(results):
    text = " ".join(doc.page_content.split())
    print(f"{index}. len={len(doc.page_content)} \n- {text[:180]}...")
    print()

**성공 기준**

- 🟢 `retriever ready` 아래에 청크가 `0.`, `1.`, `2.` 세 개 출력되고, 세 청크의 내용이 서로 다르다
- 🟢 질문을 바꿔 다시 실행하면 바뀐 질문과 관련 있는 약관 문장이 보인다

**볼 지점**
- 결과는 질문과 가까운 순서로 나옵니다. `0.`이 가장 가깝습니다.
- `len=`은 청크의 글자 수이고, 내용은 앞 180자만 보여 줍니다.
- 질문과 낱말이 똑같지 않아도 뜻이 가까운 청크가 나오는지 봅니다.

## 5. RAG 체인 <small>(예제) (과제)</small>

4장까지 만든 검색에 LLM을 이어, 약관을 근거로 답변하는 RAG 체인을 완성합니다. 검색 결과를 `context` 문자열로 합치고, 검색 → 프롬프트 → 모델 → 출력 파서를 `|`로 연결합니다. 답변은 스트리밍(조각이 생기는 대로 받는 방식)으로 출력합니다.

빈칸은 네 곳입니다. 앞의 두 곳(벡터 저장소, retriever)은 4장과 같은 일입니다. 마지막 셀에서는 `# ✍️ CHANGE_HERE:` 줄의 질문을, 프롬프트 셀에서는 system 지시문을 바꿔 봅니다.

**왜 배우나:** RAG의 Generation 단계입니다. "검색된 근거 안에서만 답하라"는 지시가 프롬프트에 어떻게 들어가고, 답변을 어떻게 바꾸는지 직접 봅니다.

준비 코드입니다. 필요한 도구를 불러오고, 약관 PDF를 읽어 청크(`chunks`)로 나눕니다. 4장 준비 코드와 같은 내용이라 실행만 하면 됩니다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
import warnings

# 도구 교체 안내 경고를 숨깁니다(실습과 관계없는 안내입니다).
warnings.filterwarnings("ignore", message=".*langchain-community.*")
from langchain_community.document_loaders import PDFMinerLoader
from langchain_chroma import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_text_splitters import RecursiveCharacterTextSplitter

from config.embeddings import get_embeddings
from config.llm import get_llm

load_dotenv()


PDF_PATH = Path("data/meritz_pet_insurance.pdf")
loader = PDFMinerLoader(str(PDF_PATH.resolve()))
documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
chunks = text_splitter.split_documents(documents)

<details><summary>강사용 메모</summary>

- 원본 `04_langchain_rag.py`(raw)를 쓰고, 노트북에서만 손봤습니다. ① `PDF_PATH` 상대 경로, 경고 숨김 ② `Chroma().delete_collection()` 추가(4장 메모와 같은 이유. 4장을 실행한 커널에서 그대로 만들면 컬렉션에 청크 100개가 쌓여 검색 결과가 중복됩니다) ③ `while True` + `input()` 반복을 질문 변수 하나로 한 번만 실행.
- raw와 솔루션은 들여쓰기·공백만 다르고, ruff 정돈 뒤에는 같습니다.
- system 지시문 CHANGE_HERE는 슬라이드의 RAG 프롬프트("맥락에 답이 없으면 '문서에서 확인할 수 없습니다'라고 말하세요")를 더해 보게 한 것입니다. 원본 지시문에는 이 규칙이 없습니다.
- LCEL(`|`)과 `RunnablePassthrough`는 1-3에서 배웁니다.

</details>

### 5.1 벡터 저장소 만들기 <small>(과제)</small>

**✍️ WRITE_HERE: 임베딩 모델을 가져오고 Chroma 벡터스토어를 생성해 주세요.**

4장에서 한 일을 다시 합니다. 청크를 임베딩해 Chroma 벡터 저장소를 만듭니다. 셀 첫 줄의 `Chroma().delete_collection()`은 저장소를 비우는 줄이라 지우지 않습니다.

**프롬프트 힌트** `get_embeddings()`로 받은 임베딩 모델과 `chunks`로 Chroma 벡터 저장소를 만들어 `embeddings`, `vectorstore`에 담아 달라고 합니다. 파일로 저장하는 옵션은 쓰지 말라고 적습니다.

**성공 기준**

- 🟢 벡터 저장소를 만드는 동안 오류가 나지 않는다
- ⛔ `NameError: name 'chunks'`가 난다 → 이 장의 준비 코드를 먼저 실행합니다

In [ ]:
# 이 셀을 다시 실행해도 같은 청크가 겹쳐 쌓이지 않게, 앞서 만든 벡터 저장소를 비웁니다.
Chroma().delete_collection()

# ✍️ WRITE_HERE: 임베딩 모델을 가져오고 Chroma 벡터스토어를 생성해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
embeddings = get_embeddings()
vectorstore = Chroma.from_documents(chunks, embedding=embeddings)
```

</details>

### 5.2 retriever로 바꾸기 <small>(과제)</small>

**✍️ WRITE_HERE: 벡터스토어를 retriever로 변환해 주세요.**

벡터 저장소를 질문과 가까운 청크 3개를 돌려주는 retriever로 바꿉니다.

**프롬프트 힌트** `vectorstore`에서 가까운 청크 3개를 찾아 주는 retriever를 만들어 `retriever`에 담아 달라고 합니다. 한 줄로 써 달라고 덧붙입니다.

**성공 기준**

- 🟢 오류 없이 셀이 끝나고 아무것도 출력되지 않는다
- ⛔ `NameError: name 'vectorstore'`가 난다 → 앞 셀을 먼저 실행합니다

In [ ]:
# ✍️ WRITE_HERE: 벡터스토어를 retriever로 변환해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
```

</details>

### 5.3 모델과 프롬프트 준비 <small>(예제)</small>

**✍️ CHANGE_HERE: system 지시문을 바꿔 봅니다**

처음에는 그대로 실행합니다. 마지막 셀까지 답변을 받아 본 뒤 이 셀로 돌아와, `# ✍️ CHANGE_HERE:` 바로 아래 줄의 지시문 끝에 "맥락에 답이 없으면 '문서에서 확인할 수 없습니다'라고 말하세요."를 더합니다. 그다음 이 셀부터 마지막 셀까지 다시 실행합니다.

In [ ]:
llm = get_llm()
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            # ✍️ CHANGE_HERE: system 지시문을 바꿔 봅니다
            "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.",
        ),
        ("human", "context:\n{context}\n\nquestion:\n{question}"),
    ]
)

### 5.4 검색 결과를 context로 합치기 <small>(과제)</small>

**✍️ WRITE_HERE: 검색된 Document 목록을 context 문자열로 합쳐 주세요.**

retriever가 돌려준 `Document` 목록을 프롬프트의 `{context}` 자리에 넣을 문자열 하나로 합칩니다. 청크 사이는 빈 줄로 구분합니다.

**프롬프트 힌트** `docs`에 든 `Document`마다 본문(`page_content`)만 꺼내, 빈 줄 하나를 사이에 두고 이어 붙인 문자열을 돌려주게 해 달라고 합니다. 함수 이름과 첫 줄은 그대로 두라고 적습니다.

**성공 기준**

- 🟢 함수를 정의하는 셀이 오류 없이 끝난다
- ⛔ `IndentationError`가 난다 → 빈칸 자리의 들여쓰기(함수 안쪽)를 맞춥니다

In [ ]:
def format_docs(docs):
    # ✍️ WRITE_HERE: 검색된 Document 목록을 context 문자열로 합쳐 주세요.
    ...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
return "\n\n".join(doc.page_content for doc in docs)
```

</details>

### 5.5 RAG 체인 연결하기 <small>(과제)</small>

**✍️ WRITE_HERE: retriever, prompt, llm, output parser를 연결해 RAG 체인을 완성해 주세요.**

retriever, 프롬프트, 모델, 출력 파서를 `|`로 이어 RAG 체인을 만듭니다. 체인에 질문 문자열 하나를 넣으면, 질문은 retriever와 `format_docs`를 거쳐 `context`가 되고, 원래 질문은 그대로 `question`이 되어 프롬프트에 들어가야 합니다.

**프롬프트 힌트** 질문 하나를 받아 검색 결과(`context`)와 질문(`question`)을 프롬프트에 채우고, 모델과 문자열 출력 파서까지 `|`로 이은 체인을 `rag_chain`에 담아 달라고 합니다. 질문을 그대로 넘기는 데는 준비 코드에서 불러온 `RunnablePassthrough`를 쓰라고 적습니다.

**성공 기준**

- 🟢 체인을 만드는 셀이 오류 없이 끝나고, 마지막 셀에서 답변이 출력된다
- ⛔ 다음 셀에서 `TypeError`나 `KeyError`가 난다 → 프롬프트에 들어갈 두 이름(`context`, `question`)을 다시 확인해 달라고 합니다

In [ ]:
# ✍️ WRITE_HERE: retriever, prompt, llm, output parser를 연결해 RAG 체인을 완성해 주세요.
...  # ✍️ 여기를 채웁니다

<details><summary>정답 보기</summary>

```python
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)
```

</details>

### 5.6 질문하고 스트리밍으로 답변 받기 <small>(예제)</small>

**✍️ CHANGE_HERE: 질문을 바꿔 봅니다**

`# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 바꿔 다시 실행합니다. 약관에 있을 법한 질문과, 약관과 관계없는 질문(예: "오늘 점심 메뉴를 추천해 줘")을 둘 다 해 봅니다.

In [ ]:
print("RAG ready")
# ✍️ CHANGE_HERE: 질문을 바꿔 봅니다
question = "펫보험 보장 내용이 뭐야?"
for message in rag_chain.stream(question):
    print(message, end="")
print("\n")

**성공 기준**

- 🟢 `RAG ready` 아래에 답변이 한 번에 나오지 않고 조금씩 이어서 출력된다
- 🟢 약관에 있을 법한 질문에는 약관 내용을 근거로 답변한다
- 🟢 system 지시문에 규칙을 더한 뒤 약관과 관계없는 질문을 하면 "문서에서 확인할 수 없습니다"라고 답변한다

**볼 지점**
- 답변이 약관 내용에 기대고 있는지 봅니다. system 지시문이 "주어진 context만 근거로 답하세요"라고 요구합니다.
- 규칙을 더하기 전과 뒤에, 약관과 관계없는 질문에 모델이 어떻게 답변하는지 비교합니다.

## 6. RAGaaS <small>(예제)</small>

RAGaaS(RAG as a Service)는 청킹·임베딩·벡터 저장소 관리를 API가 대신 해 주는 방식입니다. 앞 장에서 직접 만든 흐름(나누기 → 임베딩 → 저장 → 검색)이 API 호출 두 번(문서 업로드, 검색)으로 줄어듭니다.

이 장은 예제입니다. 여럿이 같은 인덱스(검색용 문서 묶음)를 쓰므로 문서 업로드와 삭제는 강사가 따로 하고, 코드에는 주석으로 막아 두었습니다. 검색과 답변 만들기를 실행하고, `# ✍️ CHANGE_HERE:` 줄의 질문을 바꿔 봅니다.

준비 코드의 출력이 `RAGaaS 모드: mock`이면 연습용 모의 응답입니다. 검색 결과로 `[MOCK RAGaaS]`로 시작하는 연습용 문장이 오고, 실제 약관 내용이 아니므로 답변에도 약관 내용이 나오지 않는 것이 정상입니다.

**왜 배우나:** 문서를 올리고 검색하는 일을 RAGaaS API에 맡기면 직접 만들 코드가 줄어듭니다. 직접 만든 RAG와 단계가 어떻게 대응하는지 알아 두면 둘 중 무엇을 쓸지 고를 수 있습니다.

준비 코드입니다. 필요한 도구를 불러오고, RAGaaS에 보낼 인증 정보와 인덱스 이름을 준비합니다. 키는 화면에 출력하지 않습니다. 접혀 있으니 실행만 합니다.

In [ ]:
"""
RAG as a Service (RAGaaS): 삼성 사내 RAG API 활용

기존 방식 vs RAGaaS 비교:
  기존: 청킹 → 임베딩 클래스 구현 → 벡터스토어 생성 → 검색
  RAGaaS: 문서 업로드(API 1회) → 검색(API 1회)
  청킹·임베딩·벡터스토어 관리를 API가 대신 처리한다.

사외에서는 RAG_PROVIDER=mock으로 요청·응답 형식을 연습한다.
실제 문서 내용이 필요한 실습은 사내망에서 RAG_PROVIDER=samsung으로 실행한다.

"""

import os
from pathlib import Path

from dotenv import load_dotenv
import config  # noqa: F401  .env의 COURSE_ENV(사내/사외)에 맞춰 연결 방식을 채운다

from config.llm import get_llm
from config.ragaas import post_ragaas

load_dotenv()

RAG_PROVIDER = os.getenv("RAG_PROVIDER", "samsung").strip().lower()
RAG_API_KEY = os.getenv("rag_api_key")
# 사용할 인덱스 네임
INDEX_NAME = "rp-aicoa-meritz-pet"
# 실제 사내 요청의 api-key 헤더에는 RAG_API_KEY가 들어간다.
print(f"RAGaaS 모드: {RAG_PROVIDER}")

<details><summary>강사용 메모</summary>

- **실제 RAGaaS는 사내망 전용입니다. 사외에서는 실행할 수 없습니다.** 사외는 `RAG_PROVIDER=mock`으로 요청·응답 흐름만 봅니다. mock은 검색 결과를 늘 1건 돌려주므로 `num_result_doc`을 바꿔도 결과 수가 변하지 않습니다(그래서 이 값은 CHANGE_HERE로 두지 않았습니다).
- 인증: `RAG_API_KEY` 전역변수(`rag_api_key`)와 요청 헤더 `api-key`(`config.samsung.ragaas_config`) 둘 다 있어야 합니다. 이 코드는 `post_ragaas(…, api_key=RAG_API_KEY)`로 둘을 잇습니다.
- `INDEX_NAME`은 raw의 고정값 `rp-aicoa-meritz-pet`(원본 addae97과 같음)을 씁니다. 솔루션은 `index_name` 환경변수를 씁니다.
- 업로드·삭제: raw처럼 주석으로 막았습니다(사용자 결정). 강사는 수업 전에 사내망에서 `insert-doc`을 한 번 실행해 둡니다. 서버가 청킹·임베딩까지 하므로 최대 300초 걸립니다. `delete-doc`을 누가 실행하면 모두의 검색 결과가 비니 수업 중에는 실행하지 않습니다.
- 노트북에서만 손본 곳: `실행:` 줄 삭제, `PDF_PATH` 상대 경로, TODO 주석을 설명 주석으로, `# print(result)`·`# print(context_list)`·`# ai -> …` 메모 삭제, `fields_exclude`는 솔루션처럼 켬(응답에서 큰 벡터 필드를 뺌). 삭제한 검색 요청 주석의 내용: 같은 범주의 문서를 잘 전처리했다면 같은 `INDEX_NAME`으로 여러 문서를 등록해 함께 검색할 수 있다.
- 미확인: 사내 인덱스 `rp-aicoa-meritz-pet`의 존재, `permission_groups: ['rag-public']` 권한, 실제 검색 응답 시간.

</details>

### 6.1 1부: 업로드할 문서 준비 <small>(예제)</small>

In [ ]:
# ── 1부: 문서 업로드 ──────────────────────────────────────────────────────────
# insert-doc 엔드포인트 하나로 청킹·임베딩·벡터스토어 저장을 모두 처리한다.
# chunk_factor: 청킹 방식과 크기를 서버에 위임 (직접 구현 불필요)

# 업로드할 텍스트 준비 (예: 파일을 읽어서 문자열로 전달)
PDF_PATH = Path("data/meritz_pet_insurance.pdf")

# pdfminer 로 텍스트 추출 (RAGaaS는 raw text를 받음)
from langchain_community.document_loaders import PDFMinerLoader

docs = PDFMinerLoader(str(PDF_PATH.resolve())).load()
raw_text = "\n".join(doc.page_content for doc in docs)

payload = {
    "index_name": INDEX_NAME,
    "data": {
        "doc_id": "meritz_pet_insurance",
        "title": "메리츠 펫보험 약관",
        "content": raw_text,
    },
    "chunk_factor": {
        "logic": "fixed_size",
        "chunk_size": 500,
        "chunk_overlap": 75,
        "separator": " ",
    },
}

print("=== 문서 업로드 ===")
# 문서 업로드(insert-doc)는 강사가 수업 전에 한 번 해 둡니다. 여럿이 같은 인덱스에 올리지 않도록 아래 줄은 실행하지 않습니다.
# response = post_ragaas("insert-doc", payload, api_key=RAG_API_KEY)
# print(f"상태: {response.status_code}")

### 6.2 2부: 문서 검색 <small>(예제)</small>

**✍️ CHANGE_HERE: 질문을 바꿔 봅니다**

먼저 그대로 2부와 3부를 실행합니다. 그다음 `# ✍️ CHANGE_HERE:` 바로 아래 줄의 질문을 바꾸고, 2부와 3부를 다시 실행합니다. 검색과 답변이 모두 새 질문으로 바뀌는지 봅니다.

In [ ]:
# ── 2부: 문서 검색 ────────────────────────────────────────────────────────────
# retrieve-* 엔드포인트: bm25(키워드), knn(의미), rrf(하이브리드) 중 선택

# ✍️ CHANGE_HERE: 질문을 바꿔 봅니다
question = "펫보험 보장 내용이 뭐야?"

search_payload = {
    "index_name": INDEX_NAME,
    "permission_groups": ["rag-public"],
    "query_text": question,
    "num_result_doc": 3,
    "fields_exclude": ["v_merge_title_content"],
}

print("=== 문서 검색 (RRF: 하이브리드) ===")

# retrieve-rrf 엔드포인트로 관련 문서를 검색합니다.

response = post_ragaas("retrieve-rrf", search_payload, api_key=RAG_API_KEY)
result = response.json()
hits = result["hits"]["hits"]
context_list = [hit["_source"]["merge_title_content"] for hit in hits]
context = "\n\n".join(context_list)
print(f"검색된 문서 수: {len(hits)}")

**성공 기준**

- 🟢 `=== 문서 검색 (RRF: 하이브리드) ===` 아래에 `검색된 문서 수:`와 숫자가 출력된다

### 6.3 3부: 검색 결과로 답변 만들기 <small>(예제)</small>

In [ ]:
# ── 3부: RAG 체인 ─────────────────────────────────────────────────────────────
# 검색 결과를 context로 주입해 LLM이 문서 기반으로 답변하도록 한다.
# get_llm(): 로컬은 OpenAI, 사내망은 삼성 LLM으로 자동 전환


from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

llm = get_llm()

# 모델에게 입력할 지시사항과, 사용자의 질문을 간단하게 정의
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "당신은 문서 기반 QA 어시스턴트입니다. 주어진 context만 근거로 답하세요.",
        ),
        ("human", "context:\n{context}\n\nquestion:\n{question}"),
    ]
)

chain = prompt | llm | StrOutputParser()

print(f"=== 질문: {question} ===")
for chunk in chain.stream({"context": context, "question": question}):
    print(chunk, end="", flush=True)
print("\n")

**성공 기준**

- 🟢 `=== 질문: … ===` 줄에 바꾼 질문이 보이고, 그 아래 답변이 이어서 출력된다
- 🟢 답변이 검색된 문서 내용을 근거로 한다(모의 응답이면 "문서에 없다"는 취지의 답변도 정상입니다)

### 6.4 4부: 문서 삭제(실행하지 않음) <small>(예제)</small>

In [ ]:
# ── 4부: 문서 삭제 (선택) ─────────────────────────────────────────────────────
# 실습 종료 후 업로드한 인덱스를 정리한다.

delete_payload = {
    "index_name": INDEX_NAME,
    "permission_groups": ["rag-public"],
    "doc_id": "meritz_pet_insurance",
}

# 문서 삭제(delete-doc)도 강사가 수업이 끝난 뒤 한 번만 합니다. 아래 줄은 실행하지 않습니다.
# response = post_ragaas("delete-doc", delete_payload, api_key=RAG_API_KEY)
# print(f"=== 문서 삭제: {response.status_code} ===")

**볼 지점**
- 1부의 업로드 요청(`payload`)에 `chunk_factor`가 있습니다. 청킹 방식과 크기를 서버에 맡기는 설정입니다. 앞 장에서 직접 쓴 `chunk_size`, `chunk_overlap`과 같은 값입니다.
- 2부의 검색 요청(`search_payload`)에는 인덱스 이름, 질문(`query_text`), 가져올 문서 수(`num_result_doc`)가 들어갑니다. 임베딩 코드는 보이지 않습니다.
- 검색 결과의 본문은 `hits` → `_source` → `merge_title_content` 경로로 꺼냅니다.
- 3부의 체인에는 retriever가 없습니다. 검색은 2부의 API가 이미 했습니다.

### 6.5 AI에게 물어보기

**프롬프트 힌트** 2~5장에서 직접 만든 RAG와 이 장의 RAGaaS를 나란히 놓고, 불러오기·나누기·임베딩·저장·검색·생성이 각각 어디서 일어나는지 표로 정리해 달라고 합니다. 코드는 고치지 말고, 이 노트북의 코드에 있는 내용만 근거로 하라고 적습니다.

**성공 기준**

- 🟢 RAGaaS에서는 나누기·임베딩·저장을 문서 업로드(`insert-doc`) 한 번으로 서버가 처리한다고 짚었다
- 🟢 생성(답변 만들기)은 두 방식 모두 `get_llm()`으로 준비한 모델이 한다고 정리했다
- ⛔ 코드에 없는 기능을 사실처럼 적었다 → 이 노트북 코드에서 근거가 되는 줄을 짚어 달라고 다시 묻습니다
- ⛔ 코드를 고치거나 새로 써 왔다 → 고치지 말라고 다시 적습니다

## 7. 정리

약관 PDF 한 권으로 RAG의 여섯 단계를 차례로 만들었습니다.

| 장 | 단계 | 이 노트북에서 한 일 |
|---|---|---|
| 2. PDF 불러오기와 청크 나누기 | Loading, Splitting | `PDFMinerLoader`로 읽고 `RecursiveCharacterTextSplitter`로 나눔 |
| 3. 임베딩과 유사도 | Embedding | 문장을 벡터로 바꾸고 코사인 유사도로 비교 |
| 4. 벡터 저장소와 검색 | Storage, Retrieval | Chroma에 넣고 retriever로 가까운 청크 3개 찾기 |
| 5. RAG 체인 | Generation | 검색 결과를 `context`로 넣어 답변을 스트리밍으로 받기 |
| 6. RAGaaS | 전 단계 | 업로드·검색을 API에 맡기고 답변만 만들기 |

청크 크기, 질문, system 지시문을 바꿔 다시 실행하고, 결과를 성공 기준(🟢/⛔)과 비교해 스스로 판단했습니다.